In [1]:
# Install huggingface transformers and tools
!pip install -q transformers datasets

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset

# Check if GPU is enabled
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
# 1. Load data
train_df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")

# 2. Initialize BERT Tokenizer (using DistilBERT for speed)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

# Convert DataFrames to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'target']].rename(columns={'target': 'label'}))
test_dataset = Dataset.from_pandas(test_df[['text']])

# Apply tokenization to the datasets
train_tokenized = train_dataset.map(tokenize_fn, batched=True)
test_tokenized = test_dataset.map(tokenize_fn, batched=True)
print("Tokenization completed successfully!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/7613 [00:00<?, ? examples/s]

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

Tokenization completed successfully!


In [3]:
# 3. Load pre-trained BERT model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Define fine-tuning configuration
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none"
)

# Set up the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
)

# Start training process
print("Training BERT... This will take around 1-2 minutes on GPU.")
trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training BERT... This will take around 1-2 minutes on GPU.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=476, training_loss=0.7786338870264903, metrics={'train_runtime': 108.1154, 'train_samples_per_second': 140.831, 'train_steps_per_second': 4.403, 'total_flos': 504237152984064.0, 'train_loss': 0.7786338870264903, 'epoch': 2.0})

In [4]:
# 4. Generate predictions on test data
preds = trainer.predict(test_tokenized)
final_preds = np.argmax(preds.predictions, axis=1)

# 5. Build submission file with Kaggle format
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'target': final_preds
})

submission_df.to_csv('submission.csv', index=False)
print("New advanced submission.csv created! Ready for Save Version.")

New advanced submission.csv created! Ready for Save Version.
